# Customer Segmentation — Online Retail II

Retailers need to understand how customers differ in their purchasing patterns. Grouping similar customers can help identify frequent buyers, high-spending customers, and those who may have stopped purchasing.

This experiment uses the UCI Online Retail II dataset, which contains invoice-line transactions from a UK-based online retailer between 2009 and 2011. Each row records a product on an invoice, so we aggregate purchases into one customer record using **recency**, **frequency**, and **monetary value (RFM)**. Since the data has no segment labels, this is an **unsupervised clustering** task: groups are discovered from customer behavior.

We compare **KMeans**, **Gaussian Mixture Models (GMM)**, and **HDBSCAN** using the same prepared RFM features. We review cluster profiles and internal measures to understand the customer groups and compare the methods.

## Installation

Install the libraries used to load and explore the data, prepare customer features, fit clustering models, and create plots.

In [ ]:
import sys
!uv pip install --python "{sys.executable}" -q fg-data-profiling openpyxl pandas requests scikit-learn matplotlib seaborn
%matplotlib inline

## Dataset Loading

Online Retail II contains **1,067,371** invoice-line records from a UK-based, registered online retailer, covering 1 December 2009 to 9 December 2011. The workbook has **eight fields** and no predefined clustering labels. 

UCI provides the data as an Excel workbook inside a ZIP archive, so the code below downloads and reads the official workbook directly - [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online%2Bretail%2Bii)


In [ ]:
import io
import zipfile

import pandas as pd
import requests

url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("online_retail_II.xlsx") as workbook:
        sheets = pd.read_excel(workbook, sheet_name=None)

retail_data = pd.concat(sheets.values(), ignore_index=True)
retail_data["InvoiceDate"] = pd.to_datetime(retail_data["InvoiceDate"], errors="coerce")

print(f"Loaded {len(retail_data):,} invoice lines.")


- The workbook splits the transaction history across two sheets. Concatenating them combines both periods into one dataset.
- `InvoiceDate` is converted to datetime so date-based analysis, such as monthly summaries and customer recency, works correctly.

## Dataset Overview

Each row represents one product line on an invoice. An invoice can therefore occupy several rows, and the same customer can appear across many invoices. The workbook has eight columns:

| UCI Field | Workbook Header | Type and Role |
| --- | --- | --- |
| `InvoiceNo` | `Invoice` | Nominal identifier; a leading `C` marks a cancellation |
| `StockCode` | `StockCode` | Nominal product identifier |
| `Description` | `Description` | Nominal/text feature: product name |
| `Quantity` | `Quantity` | Numeric feature: units on the invoice line |
| `InvoiceDate` | `InvoiceDate` | Date and time; UCI describes it as numeric |
| `UnitPrice` | `Price` | Numeric feature: price per unit in pounds sterling |
| `CustomerID` | `Customer ID` | Nominal customer identifier; some records have no ID |
| `Country` | `Country` | Nominal/categorical feature: customer’s country |

- **Identifiers:** `InvoiceNo`, `StockCode`, `CustomerID`
- **Numeric features:** `Quantity`, `UnitPrice`
- **Nominal features:** `Description`, `Country`
- **Date-time field:** `InvoiceDate`


In [ ]:
print("\n Dataset shape:", retail_data.shape)
print("\nFirst 5 invoice lines:")
display(retail_data.head())

print("\n Column information:")
retail_data.info()

print("\n Date range:")
print(retail_data["InvoiceDate"].min(), "to", retail_data["InvoiceDate"].max())

print("\n Distinct values:")
print("Invoices:", retail_data["Invoice"].nunique())
print("Products:", retail_data["StockCode"].nunique())
print("Customers with IDs:", retail_data["Customer ID"].nunique())
print("Countries:", retail_data["Country"].nunique())

print("\n Feature statistics:")
display(retail_data.describe(include="all").T)


## Exploratory Data Analysis (EDA)

Start by checking missing customer IDs, duplicate lines, cancellations, quantity and price values, and how transaction activity changes over time. Monthly customer counts include identified customers with non-cancellation invoices.

The full dataset has over one million rows, so the profiling report uses a reproducible 100,000-row sample to keep EDA manageable. The full dataset remains loaded for later aggregation.


In [ ]:
from data_profiling import ProfileReport
import matplotlib.pyplot as plt

# Summarize missing values by column, from highest share to lowest.
print("Missing-value share by column:")
missing_share = retail_data.isna().mean().sort_values(ascending=False)
display(missing_share.rename("missing_share").to_frame())

# Inspect cancellations, unusual quantities and prices, and duplicate rows.
invoice_numbers = retail_data["Invoice"].astype(str)
print("Rows with cancellation invoice numbers:", invoice_numbers.str.upper().str.startswith("C").sum())
print("Rows with negative quantities:", (retail_data["Quantity"] < 0).sum())
print("Rows with non-positive unit prices:", (retail_data["Price"] <= 0).sum())
print("Exact duplicate rows:", retail_data.duplicated().sum())

# Count identified customers with non-cancellation invoices by month.
is_cancellation = retail_data["Invoice"].astype(str).str.upper().str.startswith("C")
customer_purchases = retail_data.loc[
    retail_data["Customer ID"].notna()
    & ~is_cancellation
    & retail_data["InvoiceDate"].notna()
]
monthly_customers = customer_purchases.groupby(
    pd.Grouper(key="InvoiceDate", freq="MS")
)["Customer ID"].nunique()

# Plot monthly identified-customer activity.
ax = monthly_customers.plot(
    figsize=(12, 4),
    title="Identified customers with purchases by month",
)
ax.set_xlabel("Month")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

# Profile a reproducible sample to keep the report manageable.
eda_sample = retail_data.sample(n=min(100_000, len(retail_data)), random_state=42)
profile = ProfileReport(
    eda_sample,
    title="Online Retail II — EDA Sample",
    explorative=True,
)
profile.to_file("online_retail_ii_eda_report.html")


### Observations

- Missing values occur only in `Customer ID` (22.8% of rows) and `Description` (0.4%).
- Identified-customer activity is highest around **October and November**, then drops in December. December 2011 is partial, with data through the 9th.
- The data has **34,335 duplicate rows**, 22,950 negative-quantity rows, and 6,207 rows with non-positive prices.
- The **United Kingdom** accounts for about **92%** of invoice lines.
- Negative quantities may reflect **returns or cancellation records**; non-positive prices may be **adjustments or invalid entries**. UCI identifies invoices beginning with `C` as cancellations.


## Data Preparation

For a purchases-only RFM baseline, keep rows with a known customer and date, remove exact duplicates and cancellation invoices, and keep positive quantities and prices. This excludes returns from the spending totals.

In [ ]:
# Keep rows that can be assigned to a customer and ordered in time.
purchase_data = retail_data.dropna(subset=["Customer ID", "InvoiceDate"]).copy()
print(f"Rows with customer IDs and valid dates: {len(purchase_data):,}")

# Remove repeated invoice lines and cancellation invoices.
purchase_data = purchase_data.drop_duplicates()
is_cancellation = purchase_data["Invoice"].astype("string").str.upper().str.startswith("C", na=False)
purchase_data = purchase_data.loc[~is_cancellation].copy()

# Keep positive sales; returns and non-positive prices are excluded in this baseline.
is_positive_sale = (purchase_data["Quantity"] > 0) & (purchase_data["Price"] > 0)
purchase_data = purchase_data.loc[is_positive_sale].copy()

print(f"Purchase lines after cleaning: {len(purchase_data):,}")
print(f"Customers retained: {purchase_data['Customer ID'].nunique():,}")

## Customer-Level RFM Features

Calculate recency from the last purchase date, frequency from distinct invoices, and monetary value from quantity multiplied by unit price. `Customer ID` groups invoice lines into one row per customer but is not an RFM feature. Product `Description` is not used; these features summarize **when, how often, and how much each customer buys**.

In [ ]:
# Use the day after the final transaction as the common reference date.
reference_date = purchase_data["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

# Add line spend, then aggregate invoice lines to one row per customer.
rfm = (
    purchase_data.assign(LineTotal=purchase_data["Quantity"] * purchase_data["Price"])
    .groupby("Customer ID")
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("LineTotal", "sum"),
    )
)
rfm["Recency"] = (reference_date - rfm["LastPurchase"].dt.normalize()).dt.days
rfm = rfm.reset_index()[["Customer ID", "Recency", "Frequency", "Monetary"]]

print(f"RFM table: {len(rfm):,} customers")
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().T)

## Prepare RFM Features for Clustering

The first histograms show the original RFM distributions; the next show the transformed features used for clustering. Keep the original `rfm` values to interpret the customer groups.


- **`log1p`:** Compresses large values and reduces skew, so very high values have less influence on distances.
- **`StandardScaler`:** Centers each transformed feature around zero and scales it to unit variance, so the three features are on comparable scales.

In [ ]:
import matplotlib.pyplot as plt

rfm_features = ["Recency", "Frequency", "Monetary"]

# Review ranges and skew before transforming the features.
display(rfm[rfm_features].describe(percentiles=[0.5, 0.9, 0.99]).T)
display(rfm[rfm_features].skew().rename("skewness").to_frame())

# Plot each feature to inspect its distribution and extreme values.
axes = rfm[rfm_features].hist(bins=40, figsize=(12, 4))
x_axis_labels = {
    "Recency": "Days since last purchase",
    "Frequency": "Distinct invoices",
    "Monetary": "Total spend (£)",
}
for ax, feature in zip(axes.ravel(), rfm_features):
    ax.set_title(feature)
    ax.set_xlabel(x_axis_labels[feature])
    ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

- Frequency and Monetary are strongly right-skewed, with most customers at lower values and a smaller number at very high values. Recency is less skewed.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Compress large positive values, then put all features on a comparable scale.
rfm_log = np.log1p(rfm[rfm_features])
rfm_scaler = StandardScaler()
X_scaled = pd.DataFrame(
    rfm_scaler.fit_transform(rfm_log),
    columns=rfm_features,
    index=rfm.index,
)

display(rfm_log.skew().rename("skewness_after_log").to_frame())
display(X_scaled.head())

# Check the scaled feature distributions used by KMeans.
axes = X_scaled.hist(bins=40, figsize=(12, 4))
for ax, feature in zip(axes.ravel(), rfm_features):
    ax.set_title(feature)
    ax.set_xlabel("Standardized value")
    ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

- After `log1p`, Recency and Monetary are more balanced; Frequency remains somewhat right-skewed. Standardization changes the scales, not the distribution shapes.

## 1. KMeans

### Choose the Number of Clusters

Compare candidate values of `k` using inertia and silhouette. These are internal guides because the data has no known segment labels. The models use all customers; silhouette is estimated on a reproducible sample of up to 2,000 customers.

**Note:**

- **Inertia:** Measures how close customers are to their assigned cluster centers. It falls as more clusters are added.
- **Elbow method:** Uses the inertia plot to find where adding clusters gives much smaller improvements; this helps suggest a reasonable `k`.
- **Silhouette:** Measures how well-separated the clusters are. It ranges from -1 to 1; higher scores generally indicate clearer separation.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = range(2, 9)
silhouette_sample_size = min(2_000, len(X_scaled))
cluster_scores = []

# Fit each candidate and record its within-cluster spread and separation.
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    cluster_scores.append(
        {
            "k": k,
            "inertia": kmeans.inertia_,
            "silhouette": silhouette_score(
                X_scaled,
                labels,
                sample_size=silhouette_sample_size,
                random_state=42,
            ),
        }
    )

kmeans_scores = pd.DataFrame(cluster_scores)
display(kmeans_scores)

# Compare the elbow in inertia with the silhouette scores.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(kmeans_scores["k"], kmeans_scores["inertia"], marker="o")
axes[0].set_title("Inertia by cluster count")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].set_xticks(list(k_values))

axes[1].plot(kmeans_scores["k"], kmeans_scores["silhouette"], marker="o")
axes[1].set_title("Silhouette by cluster count")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_ylim(-1, 1)
axes[1].set_xticks(list(k_values))

plt.tight_layout()
plt.show()

- Inertia keeps falling, with smaller gains after about `k=4`. Silhouette is highest at `k=2` (**0.435**); `k=4` scores **0.366**, making it a more detailed option with somewhat weaker separation.

### Fit and Profile the Cluster Model

Use `k=4` as an initial customer segmentation. The silhouette comparison favored `k=2`, while four clusters may provide more detailed customer profiles.

In [ ]:
from sklearn.cluster import KMeans

# Fit the four-cluster model on the scaled RFM features.
selected_k = 4
final_kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init=10)
rfm_clustered = rfm.copy()
rfm_clustered["Cluster"] = final_kmeans.fit_predict(X_scaled)

# Summarize group sizes and medians using the original RFM values.
cluster_profile = (
    rfm_clustered.groupby("Cluster")
    .agg(
        Customers=("Customer ID", "size"),
        Median_recency_days=("Recency", "median"),
        Median_invoices=("Frequency", "median"),
        Median_spend=("Monetary", "median"),
    )
    .rename(
        columns={
            "Median_recency_days": "Median recency (days)",
            "Median_invoices": "Median invoices",
            "Median_spend": "Median spend (£)",
        }
    )
)
cluster_profile["Customer share (%)"] = (
    100 * cluster_profile["Customers"] / len(rfm_clustered)
)
cluster_profile = cluster_profile[
    [
        "Customers",
        "Customer share (%)",
        "Median recency (days)",
        "Median invoices",
        "Median spend (£)",
    ]
]
display(cluster_profile.round(1))


- Review cluster sizes and median RFM values in their original units before deciding whether the four groups are useful. Cluster numbers are labels, not rankings.

### Cluster Diagnostics

These plots help us understand the clusters. They show group sizes, how customer RFM patterns differ, and how clearly the groups are separated.

#### Cluster Sizes

Check how **customers are distributed** across the assigned clusters.

In [ ]:
import matplotlib.pyplot as plt

# Plot customer counts and shares for each cluster.
cluster_sizes = cluster_profile["Customers"]
ax = cluster_sizes.plot(kind="bar", figsize=(6, 4), color="steelblue")

for position, cluster in enumerate(cluster_sizes.index):
    count = cluster_sizes.loc[cluster]
    share = cluster_profile.loc[cluster, "Customer share (%)"]
    ax.text(position, count, f"{count:,} ({share:.1f}%)", ha="center", va="bottom")

ax.set_title(f"Customers by cluster (k={selected_k})")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of customers")
ax.set_xticklabels([f"Cluster {cluster}" for cluster in cluster_sizes.index], rotation=0)

plt.tight_layout()
plt.show()

- The bars show the number and share of customers in each cluster.
- Cluster 3 is largest (**33.3%**) and Cluster 0 is smallest (**20.5%**); all four groups are substantial.

#### RFM Profile Heatmap

The heatmap shows each **cluster center** in scaled RFM space. Positive and negative values indicate features above and below the overall average, making the three measures comparable.

In [ ]:
import seaborn as sns

center_profile = pd.DataFrame(
    final_kmeans.cluster_centers_,
    columns=rfm_features,
    index=[f"Cluster {cluster}" for cluster in range(selected_k)],
)

# Compare relative RFM levels across clusters.
plt.figure(figsize=(7, 3))
sns.heatmap(
    center_profile,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    cbar_kws={"label": "Standardized RFM value"},
)
plt.title(f"RFM profile by cluster (k={selected_k})")
plt.xlabel("RFM feature")
plt.ylabel("")
plt.tight_layout()
plt.show()

- Rows are clusters and columns are RFM features. Red means above average; blue means below. Higher Recency means longer inactivity.
- Cluster 0 has very recent purchases and high frequency and spend. Cluster 3 shows the reverse pattern.

#### Silhouette by Cluster

Inspect customer-level separation scores on a reproducible sample of up to 2,000 customers. Values near zero or below suggest overlap or possible misassignment.

In [ ]:
from sklearn.metrics import silhouette_samples

silhouette_sample_size = min(2_000, len(X_scaled))
silhouette_sample_index = X_scaled.sample(
    n=silhouette_sample_size, random_state=42
).index
sample_features = X_scaled.loc[silhouette_sample_index]
sample_clusters = rfm_clustered.loc[silhouette_sample_index, "Cluster"]
sample_silhouette = silhouette_samples(sample_features, sample_clusters)
silhouette_data = pd.DataFrame(
    {"Cluster": sample_clusters.to_numpy(), "Silhouette": sample_silhouette}
)

# Plot sorted customer scores in separate bands for each cluster.
fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for cluster in sorted(silhouette_data["Cluster"].unique()):
    values = np.sort(
        silhouette_data.loc[
            silhouette_data["Cluster"] == cluster, "Silhouette"
        ].to_numpy()
    )
    y_upper = y_lower + len(values)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, values, alpha=0.7)
    ax.text(-0.95, y_lower + len(values) / 2, f"Cluster {cluster}", va="center")
    y_lower = y_upper + 10

mean_silhouette = silhouette_data["Silhouette"].mean()
ax.axvline(
    mean_silhouette,
    color="red",
    linestyle="--",
    label=f"Sample mean: {mean_silhouette:.3f}",
)
ax.set_xlim(-1, 1)
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Sampled customers, sorted by score")
ax.set_title(f"Silhouette scores by cluster (k={selected_k})")
ax.legend()
plt.tight_layout()
plt.show()

- Each colored band is a cluster. Scores farther right mean a clearer fit; scores near zero show overlap. The dashed line is the average.
- The four-cluster mean is **0.366**. Some customers are near zero, and the two-cluster score was higher at **0.435**.

#### RFM Distributions by Cluster

Compare the original RFM values by cluster. The logarithmic y-axes make the wide ranges easier to view; individual outlier points are hidden for readability.

In [ ]:
# Plot original-unit RFM distributions with a log scale for each feature.
cluster_ids = sorted(rfm_clustered["Cluster"].unique())
axis_labels = {
    "Recency": "Days since last purchase",
    "Frequency": "Distinct invoices",
    "Monetary": "Total spend (£)",
}
fig, axes = plt.subplots(1, len(rfm_features), figsize=(14, 4))
for ax, feature in zip(axes, rfm_features):
    values_by_cluster = [
        rfm_clustered.loc[rfm_clustered["Cluster"] == cluster, feature]
        for cluster in cluster_ids
    ]
    ax.boxplot(
        values_by_cluster,
        tick_labels=[f"Cluster {cluster}" for cluster in cluster_ids],
        showfliers=False,
    )
    ax.set_yscale("log")
    ax.set_title(feature)
    ax.set_xlabel("Cluster")
    ax.set_ylabel(axis_labels[feature])

fig.suptitle(f"RFM distributions by cluster (k={selected_k}; log y-axis)")
plt.tight_layout()
plt.show()

- The panels compare Recency, Frequency, and Monetary across clusters. Lines show medians; boxes show the middle half. The y-axes are logarithmic, with outliers hidden.
- Cluster 0 has the highest median frequency and spend. Cluster 3 is the least active and lowest spending; Clusters 1 and 2 sit between them.

## 2. Gaussian Mixture Model (GMM)

GMM assigns every customer to a component and gives membership probabilities. It uses the same scaled RFM features as the other methods.

### Choose the Number of Components

Compare 2–8 components at two covariance regularization settings (`reg_covar=0.01` and `0.1`) using BIC; lower BIC is preferred within each setting. The plot shows how BIC changes as components are added. A minimum helps choose a count, while the curve shape shows whether the tested range is sufficient.

In [ ]:
from sklearn.mixture import GaussianMixture

gmm_reg_covar_values = [1e-2, 1e-1]
gmm_component_values = range(2, 9)
gmm_score_rows = []

# Check how the BIC curve changes as the covariance floor increases.
for reg_covar in gmm_reg_covar_values:
    for n_components in gmm_component_values:
        candidate_gmm = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            n_init=5,
            reg_covar=reg_covar,
            random_state=42,
        )
        candidate_gmm.fit(X_scaled)
        min_covariance_eigenvalue = np.linalg.eigvalsh(
            candidate_gmm.covariances_
        ).min()
        gmm_score_rows.append(
            {
                "reg_covar": reg_covar,
                "Components": n_components,
                "BIC": candidate_gmm.bic(X_scaled),
                "Converged": candidate_gmm.converged_,
                "Minimum covariance eigenvalue": min_covariance_eigenvalue,
            }
        )

gmm_scores = pd.DataFrame(gmm_score_rows)
display(gmm_scores.round(4))

# Plot a separate BIC curve for each covariance floor.
plt.figure(figsize=(7, 4))
for reg_covar, scores in gmm_scores.groupby("reg_covar"):
    plt.plot(
        scores["Components"],
        scores["BIC"],
        marker="o",
        label=f"reg_covar={reg_covar:g}",
    )
plt.title("GMM BIC by component count and covariance floor")
plt.xlabel("Number of components")
plt.ylabel("BIC (lower is better within each curve)")
plt.xticks(list(gmm_component_values))
plt.legend()
plt.tight_layout()
plt.show()

- **GMM:** Groups customers into **oval-shaped** patterns and gives membership probabilities.
- **`reg_covar`:** Sets a minimum group **spread** to help prevent overly narrow, overfit groups.
- **BIC:** Balances model fit and **number of groups**; lower is better within the same `reg_covar` setting.
- With `reg_covar=0.1`, BIC is lowest at 7 components and rises at 8. With `0.01`, it keeps falling through 8, so that setting needs a wider search.

### Fit and Profile the Selected GMM

The component count and `reg_covar` are GMM settings (hyperparameters). This step uses `reg_covar=0.1` and the component count selected by BIC, then summarizes the customer groups.

In [ ]:
# Choose the component count with the lowest BIC at the primary covariance floor.
GMM_REG_COVAR = 1e-1
primary_gmm_scores = gmm_scores.loc[gmm_scores["reg_covar"] == GMM_REG_COVAR]
selected_gmm_components = int(
    primary_gmm_scores.loc[primary_gmm_scores["BIC"].idxmin(), "Components"]
)

gmm_model = GaussianMixture(
    n_components=selected_gmm_components,
    covariance_type="full",
    n_init=5,
    reg_covar=GMM_REG_COVAR,
    random_state=42,
)
gmm_model.fit(X_scaled)

# Assign each customer to its most likely component and keep that probability.
gmm_probabilities = gmm_model.predict_proba(X_scaled)
rfm_gmm = rfm.copy()
rfm_gmm["Cluster"] = gmm_model.predict(X_scaled)
rfm_gmm["Assignment confidence"] = gmm_probabilities.max(axis=1)

# Summarize component sizes, original-unit RFM medians, and assignment confidence.
gmm_profile = (
    rfm_gmm.groupby("Cluster")
    .agg(
        Customers=("Customer ID", "size"),
        Median_recency_days=("Recency", "median"),
        Median_invoices=("Frequency", "median"),
        Median_spend=("Monetary", "median"),
        Mean_assignment_confidence=("Assignment confidence", "mean"),
    )
    .rename(
        columns={
            "Median_recency_days": "Median recency (days)",
            "Median_invoices": "Median invoices",
            "Median_spend": "Median spend (£)",
            "Mean_assignment_confidence": "Mean assignment confidence",
        }
    )
)

# Check the smallest covariance eigenvalue for each component.
component_min_eigenvalues = pd.Series(
    np.linalg.eigvalsh(gmm_model.covariances_).min(axis=1),
    index=range(selected_gmm_components),
)
gmm_profile["Minimum covariance eigenvalue"] = component_min_eigenvalues.reindex(
    gmm_profile.index
)
gmm_profile["Customer share (%)"] = 100 * gmm_profile["Customers"] / len(rfm_gmm)

# Keep the summary columns together and display the selected model results.
gmm_profile = gmm_profile[
    [
        "Customers",
        "Customer share (%)",
        "Median recency (days)",
        "Median invoices",
        "Median spend (£)",
        "Mean assignment confidence",
        "Minimum covariance eigenvalue",
    ]
]
print(f"Selected components by BIC: {selected_gmm_components}")
print(f"Covariance regularization: {GMM_REG_COVAR:g}")
print(f"Converged: {gmm_model.converged_} ({gmm_model.n_iter_} iterations)")
display(gmm_profile.round(4))

- With `reg_covar=0.1`, the seven-component fit includes a small, very frequent, high-spending group and a large long-inactive, low-spending group.
- **`fit`:** Learns the groups from the scaled RFM data.
- **`predict`:** Assigns each customer to the most likely group.
- **`predict_proba`:** Gives the probability for each group; the highest is saved as assignment confidence.
- In the profile table, customer count and RFM values describe each group. Mean assignment confidence shows how sure GMM is about its assignments; the minimum covariance eigenvalue checks whether a group has become too narrow.

### Cluster Diagnostics

These plots help us understand the GMM components. They show group sizes, how customer RFM patterns differ, and how clearly the groups are separated.

#### Cluster Sizes

This plot shows the number and share of customers assigned to each component.

In [ ]:
# Plot customer counts and shares for each GMM component.
gmm_sizes = gmm_profile["Customers"]
ax = gmm_sizes.plot(kind="bar", figsize=(6, 4), color="steelblue")

for position, cluster in enumerate(gmm_sizes.index):
    count = gmm_sizes.loc[cluster]
    share = gmm_profile.loc[cluster, "Customer share (%)"]
    ax.text(position, count, f"{count:,} ({share:.1f}%)", ha="center", va="bottom")

ax.set_title(f"GMM customers by component (n={selected_gmm_components})")
ax.set_xlabel("Component")
ax.set_ylabel("Number of customers")
ax.set_xticklabels([f"Ctr {cluster}" for cluster in gmm_sizes.index], rotation=0)

plt.tight_layout()
plt.show()

- The bars show the number and share of customers in each GMM component.
- Component 4 is largest (**26.9%**); Component 5 is smallest (**2.6%**) and represents the very frequent, high-spending group.

#### Relative RFM Profiles

This heatmap shows each component’s average RFM values on the standardized scale; positive values are above the overall average.

In [ ]:
# Compare component means in standardized RFM space.
gmm_mean_profile = pd.DataFrame(
    gmm_model.means_,
    columns=rfm_features,
    index=[f"Component {cluster}" for cluster in range(selected_gmm_components)],
)
plt.figure(figsize=(7, max(2.5, 0.7 * selected_gmm_components)))
sns.heatmap(
    gmm_mean_profile,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    cbar_kws={"label": "Standardized RFM value"},
)
plt.title("GMM component means in scaled RFM space")
plt.xlabel("RFM feature")
plt.ylabel("")
plt.tight_layout()
plt.show()

- Rows are components and columns are RFM features. Red means above average; blue means below. Higher Recency means longer inactivity.
- Component 5 has the strongest Frequency and Monetary values and very low Recency. Component 4 shows the opposite pattern.

#### Silhouette by Component

This plot shows sampled customer separation scores for the GMM assignments. Scores near zero or below indicate overlap.

In [ ]:
from sklearn.metrics import silhouette_samples

# Inspect hard-assignment separation on a reproducible sample.
gmm_silhouette_sample_size = min(2_000, len(X_scaled))
gmm_sample_index = X_scaled.sample(
    n=gmm_silhouette_sample_size, random_state=42
).index
gmm_sample_features = X_scaled.loc[gmm_sample_index]
gmm_sample_clusters = rfm_gmm.loc[gmm_sample_index, "Cluster"]
gmm_sample_silhouette = silhouette_samples(gmm_sample_features, gmm_sample_clusters)
gmm_silhouette_data = pd.DataFrame(
    {"Cluster": gmm_sample_clusters.to_numpy(), "Silhouette": gmm_sample_silhouette}
)

# Plot sorted customer scores in separate component bands.
fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for cluster in sorted(gmm_silhouette_data["Cluster"].unique()):
    values = np.sort(
        gmm_silhouette_data.loc[
            gmm_silhouette_data["Cluster"] == cluster, "Silhouette"
        ].to_numpy()
    )
    y_upper = y_lower + len(values)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, values, alpha=0.7)
    ax.text(-0.95, y_lower + len(values) / 2, f"Component {cluster}", va="center")
    y_lower = y_upper + 10

gmm_mean_silhouette = gmm_silhouette_data["Silhouette"].mean()
ax.axvline(
    gmm_mean_silhouette,
    color="red",
    linestyle="--",
    label=f"Sample mean: {gmm_mean_silhouette:.3f}",
)
ax.set_title(f"GMM silhouette by component (n={selected_gmm_components})")
ax.set_xlabel("Silhouette score")
ax.set_ylabel("Customers (sorted within component)")
ax.set_xlim(-1, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

- Each colored band shows customer scores for one component. Scores farther right mean clearer separation; scores near zero show overlap. The dashed line is the average.
- The sampled mean is **0.314**, below KMeans `k=4` at **0.366**, indicating weaker separation for this GMM fit.

#### RFM Distributions by Component

These boxplots compare the original Recency, Frequency, and Monetary values across components.

In [ ]:
# Compare original-unit RFM distributions; log y-axes show wide ranges.
gmm_cluster_ids = sorted(rfm_gmm["Cluster"].unique())
gmm_axis_labels = {
    "Recency": "Days since last purchase",
    "Frequency": "Distinct invoices",
    "Monetary": "Total spend (£)",
}

fig, axes = plt.subplots(1, len(rfm_features), figsize=(14, 4))
for ax, feature in zip(axes, rfm_features):
    values_by_cluster = [
        rfm_gmm.loc[rfm_gmm["Cluster"] == cluster, feature]
        for cluster in gmm_cluster_ids
    ]

    ax.boxplot(
        values_by_cluster,
        tick_labels=[f"Component {cluster}" for cluster in gmm_cluster_ids],
        showfliers=False,
    )
    
    ax.set_yscale("log")
    ax.set_title(feature)
    ax.set_xlabel("Component")
    ax.set_ylabel(gmm_axis_labels[feature])
    ax.tick_params(axis="x", rotation=45)

fig.suptitle(f"RFM distributions by GMM component (n={selected_gmm_components}; log y-axis)")
plt.tight_layout()
plt.show()

- The panels compare original Recency, Frequency, and Monetary values across components. Lines show medians; boxes show the middle half. The y-axes are logarithmic, with outliers hidden.
- Component 5 has a median of 40 invoices and £20,395 spend, with 4-day recency. Component 4 has 429-day recency, one invoice, and £244 spend.

## 3. HDBSCAN

HDBSCAN finds clusters from the density of the scaled RFM features, so it does not require a chosen `k`. It can label customers that do not fit a dense group as noise (`-1`).

### Fit and Profile HDBSCAN

Fit the model with `min_cluster_size=50` and `min_samples=10`, then review the cluster sizes, customer profiles, and share labeled as noise.

In [ ]:
from sklearn.cluster import HDBSCAN

# Set HDBSCAN parameters and reuse the scaled RFM features.
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10)
hdbscan_labels = hdbscan_model.fit_predict(X_scaled)

# Attach each customer’s cluster label; -1 marks noise.
rfm_hdbscan = rfm.copy()
rfm_hdbscan["Cluster"] = hdbscan_labels

# Summarize cluster sizes and median RFM values in original units.
hdbscan_profile = (
    rfm_hdbscan.groupby("Cluster")
    .agg(
        Customers=("Customer ID", "size"),
        Median_recency_days=("Recency", "median"),
        Median_invoices=("Frequency", "median"),
        Median_spend=("Monetary", "median"),
    )
    .rename(
        columns={
            "Median_recency_days": "Median recency (days)",
            "Median_invoices": "Median invoices",
            "Median_spend": "Median spend (£)",
        }
    )
)

# Calculate each group’s share of all customers.
hdbscan_profile["Customer share (%)"] = (
    100 * hdbscan_profile["Customers"] / len(rfm_hdbscan)
)
hdbscan_profile = hdbscan_profile[
    [
        "Customers",
        "Customer share (%)",
        "Median recency (days)",
        "Median invoices",
        "Median spend (£)",
    ]
]

# Report the number of clusters and the share labeled as noise.
hdbscan_n_clusters = rfm_hdbscan.loc[rfm_hdbscan["Cluster"] != -1, "Cluster"].nunique()
hdbscan_noise_share = 100 * (rfm_hdbscan["Cluster"] == -1).mean()
print(f"Clusters found: {hdbscan_n_clusters}")
print(f"Customers labeled as noise: {hdbscan_noise_share:.1f}%")
display(hdbscan_profile.round(1))

- **`min_cluster_size=50`:** Groups smaller than 50 customers are treated as noise.
- **`min_samples=10`:** Controls how dense a group must be; larger values usually mark more customers as noise.
- HDBSCAN found **5 clusters** and labeled **19.3%** of customers as noise.
- The noise group is relatively recent, with median recency of 34 days, 4 invoices, and £1,635 spend; it is not simply a low-value group.

### Cluster Diagnostics

These plots help us understand the HDBSCAN clusters. They show group sizes, how customer RFM patterns differ, and how many customers remain unassigned as noise.

#### HDBSCAN Cluster Sizes

This plot shows the size of each cluster and the customers labeled as noise.

In [ ]:
# Include noise in the count plot so unassigned customers remain visible.
hdbscan_sizes = rfm_hdbscan["Cluster"].value_counts().sort_index()
hdbscan_labels = [
    "Noise (-1)" if cluster == -1 else f"Cluster {cluster}"
    for cluster in hdbscan_sizes.index
]
ax = hdbscan_sizes.plot(kind="bar", figsize=(7, 4), color="steelblue")
for position, (cluster, count) in enumerate(hdbscan_sizes.items()):
    share = 100 * count / len(rfm_hdbscan)
    ax.text(position, count, f"{count:,} ({share:.1f}%)", ha="center", va="bottom")
ax.set_title("Customers by HDBSCAN group")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of customers")
ax.set_xticklabels(hdbscan_labels, rotation=0)
plt.tight_layout()
plt.show()

- The bars show the number and **share of customers** in each cluster, including noise.
- HDBSCAN found **five clusters**. Cluster 3 is largest (**35.3%**), Cluster 2 is smallest (**2.4%**), and **19.3%** of customers are labeled as noise.

#### RFM Distributions by HDBSCAN Group

These boxplots compare original RFM values across the clusters and the noise group.

In [ ]:
# Compare original-unit RFM values; noise is shown as its own diagnostic group.
hdbscan_cluster_ids = sorted(rfm_hdbscan["Cluster"].unique())
hdbscan_axis_labels = {
    "Recency": "Days since last purchase",
    "Frequency": "Distinct invoices",
    "Monetary": "Total spend (£)",
}
fig, axes = plt.subplots(1, len(rfm_features), figsize=(14, 4))
for ax, feature in zip(axes, rfm_features):
    values_by_cluster = [
        rfm_hdbscan.loc[rfm_hdbscan["Cluster"] == cluster, feature]
        for cluster in hdbscan_cluster_ids
    ]
    ax.boxplot(
        values_by_cluster,
        tick_labels=["Noise" if cluster == -1 else f"Cluster {cluster}" for cluster in hdbscan_cluster_ids],
        showfliers=False,
    )
    ax.set_yscale("log")
    ax.set_title(feature)
    ax.set_xlabel("HDBSCAN group")
    ax.set_ylabel(hdbscan_axis_labels[feature])
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("RFM distributions by HDBSCAN group (log y-axis)")
plt.tight_layout()
plt.show()

- The panels compare original RFM values across clusters and the noise group. Lines show medians; boxes show the middle half. The y-axes are logarithmic.
- Cluster 1 resembles KMeans Cluster 3: both are long-inactive, low-spend groups. Cluster 3 contains active repeat buyers. The **noise group** is relatively recent and moderate-value, not simply low-value outliers.

## Summary

Using Online Retail II purchase history, this notebook explores customer segmentation. Each customer is described by how recently they bought, how often they ordered, and how much they spent. We compare three ways to group customers:

- **KMeans (4 clusters):** Works best for **roughly round groups of similar size**. You **choose the number of clusters**, and everyone is assigned. Four clusters give more detail, while two clusters scored better on separation. Use it when you need a straightforward segment for every customer.
- **GMM (7 clusters at `reg_covar=0.1`):** Can model **oval-shaped groups** and estimate **membership probabilities**. It found a small group of **very frequent, high-spending customers**, though separation was weaker than KMeans. The `0.01` run had lower BIC but was still improving at the eight-cluster limit, so seven is provisional.
- **HDBSCAN (5 clusters):** Finds **dense groups without choosing the number in advance**. It left about **19% of customers unassigned**. Use it when the number of groups is unknown and some customers can remain unassigned.
- **Overall observation:** All three methods found **long-inactive, low-spending customers**, but divided active customers differently. **KMeans with four clusters** is a practical starting point when every customer needs a segment. Since there are no known correct labels, keep groups that support useful customer actions.